In [18]:
import pandas as pd, json, re, os
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

PATH = ".././data/train.csv"
df = pd.read_csv(PATH, sep=';')
print("Rows:", len(df))
df.head(200)[["sample","annotation"]]


Rows: 27251


,sample,annotation
0,aa,"[(0, 2, 'O')]"
1,aala,"[(0, 4, 'O')]"
2,aarcca,"[(0, 6, 'O')]"
3,abon,"[(0, 4, 'O')]"
4,abso,"[(0, 4, 'B-BRAND')]"
...,...,...
195,backer,"[(0, 6, 'B-BRAND')]"
196,badyart,"[(0, 7, 'B-BRAND')]"
197,baffy,"[(0, 5, 'B-BRAND')]"
198,bager,"[(0, 5, 'B-BRAND')]"


In [19]:
def parse_annotation_cell(cell):
    if pd.isna(cell): return []
    s = str(cell).strip()
    # Try json
    try:
        parsed = json.loads(s.replace("'", '"'))
        return parsed
    except Exception:
        pass
    try:
        parsed = eval(s)
        return parsed
    except Exception:
        # fallback regex parse (start,end,label)
        matches = re.findall(r"\(?\s*(\d+)\s*,\s*(\d+)\s*,\s*'?(B-|I-)?([A-Z]+)'?\s*\)?", s)
        spans = []
        for start,end,bpre,typ in matches:
            label = f"{bpre or ''}{typ}".strip() if bpre else typ
            spans.append((int(start), int(end), label))
        return spans

def spans_to_entities_text(text, spans):
    ents = []
    for s in spans:
        if len(s) >= 3:
            start,end,label = s[0], s[1], s[2]
        else:
            continue
        lab = label.split("-",1)[-1] if "-" in label else label
        ent_text = text[start:end].strip()
        ents.append((start,end,lab,ent_text))
    return ents

# Parse all annotations
parsed_spans = []
entities_all = []
for _, row in df.iterrows():
    txt = str(row['sample'])
    spans = parse_annotation_cell(row['annotation'])
    parsed_spans.append(spans)
    ents = spans_to_entities_text(txt, spans)
    entities_all.extend(ents)

df['parsed_spans'] = parsed_spans
df['entities'] = df.apply(lambda r: spans_to_entities_text(str(r['sample']), r['parsed_spans']), axis=1)
df['n_entities'] = df['entities'].apply(len)
df['n_chars'] = df['sample'].str.len()

df.head(200)[["sample","annotation","parsed_spans","entities","n_entities"]]


,sample,annotation,parsed_spans,entities,n_entities
0,aa,"[(0, 2, 'O')]","[(0, 2, O)]","[(0, 2, O, aa)]",1
1,aala,"[(0, 4, 'O')]","[(0, 4, O)]","[(0, 4, O, aala)]",1
2,aarcca,"[(0, 6, 'O')]","[(0, 6, O)]","[(0, 6, O, aarcca)]",1
3,abon,"[(0, 4, 'O')]","[(0, 4, O)]","[(0, 4, O, abon)]",1
4,abso,"[(0, 4, 'B-BRAND')]","[(0, 4, B-BRAND)]","[(0, 4, BRAND, abso)]",1
...,...,...,...,...,...
195,backer,"[(0, 6, 'B-BRAND')]","[(0, 6, B-BRAND)]","[(0, 6, BRAND, backer)]",1
196,badyart,"[(0, 7, 'B-BRAND')]","[(0, 7, B-BRAND)]","[(0, 7, BRAND, badyart)]",1
197,baffy,"[(0, 5, 'B-BRAND')]","[(0, 5, B-BRAND)]","[(0, 5, BRAND, baffy)]",1
198,bager,"[(0, 5, 'B-BRAND')]","[(0, 5, B-BRAND)]","[(0, 5, BRAND, bager)]",1


In [24]:
df[df['n_entities'] == 6]

,sample,annotation,parsed_spans,entities,n_entities,n_chars
10793,круассаны с шоколадом 3 хлеб завод,"[(0, 9, 'B-TYPE'), (10, 11, 'O'), (12, 21, 'O'...","[(0, 9, B-TYPE), (10, 11, O), (12, 21, O), (22...","[(0, 9, TYPE, круассаны), (10, 11, O, с), (12,...",6,34
13304,молочный коктейль иди смотри мам сделаю,"[(0, 8, 'B-TYPE'), (9, 17, 'I-TYPE'), (18, 21,...","[(0, 8, B-TYPE), (9, 17, I-TYPE), (18, 21, O),...","[(0, 8, TYPE, молочный), (9, 17, TYPE, коктейл...",6,39
22142,сухой корм для кошек van kot,"[(0, 5, 'B-TYPE'), (6, 10, 'I-TYPE'), (11, 14,...","[(0, 5, B-TYPE), (6, 10, I-TYPE), (11, 14, O),...","[(0, 5, TYPE, сухой), (6, 10, TYPE, корм), (11...",6,28


In [31]:
# Global stats
total_sentences = len(df)
total_entities = df['n_entities'].sum()
print("Total sentences:", total_sentences)
print("Total entities:", total_entities)
print("Sentences with >=1 entity:", (df['n_entities']>0).sum())

# Entity type counts and common entity texts
type_counter = Counter([ent[2] for ent in entities_all])
text_counter_by_type = defaultdict(Counter)
for s,e,t,txt in entities_all:
    text_counter_by_type[t][txt.lower()] += 1

print("\nEntity type counts:")
for t,c in type_counter.most_common():
    print(f"  {t}: {c}")


Total sentences: 27251
Total entities: 42253
Sentences with >=1 entity: 27251

Entity type counts:
  TYPE: 29060
  BRAND: 7699
  O: 5379
  VOLUME: 84
  PERCENT: 30
  0: 1


In [34]:

with open('en4class.txt', 'w', encoding='utf-8') as f:
    for t in sorted(text_counter_by_type):
        print(f"\nTop entities for {t}:")
        f.write(f"\nTop entities for {t}:")
        for ent,cnt in text_counter_by_type[t].most_common(20):
            print(f"  {ent!r}: {cnt}")
            f.write(f"  {ent!r}: {cnt}")



Top entities for 0:
  'абрикос': 1

Top entities for BRAND:
  'artfruit': 52
  'нф': 49
  'тендер': 47
  'красная': 37
  'производство': 36
  'фруто': 34
  'цена': 33
  'маркет': 32
  'няня': 31
  'пр!ст': 28
  'econta': 26
  'стм': 23
  'сады': 22
  'тенде': 21
  'перекресток': 21
  'агуша': 19
  'производств': 19
  'тенд': 18
  'беллини': 18
  'производст': 16

Top entities for O:
  'для': 781
  'с': 430
  'в': 192
  'без': 115
  'доя': 70
  'из': 64
  'на': 45
  'по': 43
  'сахара': 38
  'со': 35
  'посуды': 33
  'к': 26
  'стирки': 26
  'мытья': 26
  'кошек': 24
  'от': 24
  'собак': 23
  'например': 18
  'тебе': 18
  'курицы': 15

Top entities for PERCENT:
  '%': 4
  '1': 2
  '5%': 2
  '10': 2
  '10%': 2
  '20': 2
  '33': 2
  '15%': 2
  '9': 2
  '0': 1
  '1%': 1
  '72': 1
  '82': 1
  '3,2': 1
  '3.2': 1
  '20%': 1
  '33%': 1
  '3': 1
  '25': 1

Top entities for TYPE:
  'сыр': 268
  'хлеб': 157
  'сок': 132
  'вода': 132
  'корм': 127
  'чай': 113
  'колбаса': 106
  'приправа': 10

In [ ]:

# Sequence length distributions (chars)
plt.figure(figsize=(8,4))
df['n_chars'].hist(bins=50)
plt.title("Histogram: characters per sample")
plt.xlabel("Characters")
plt.ylabel("Number of samples")
plt.show()

# Entities per sample distribution
plt.figure(figsize=(8,4))
df['n_entities'].hist(bins=10)
plt.title("Histogram: entities per sample")
plt.xlabel("Number of entities in sample")
plt.ylabel("Number of samples")
plt.show()

# Entity length distributions (chars and tokens)
entity_lens_chars = [len(txt) for s,e,t,txt in entities_all]
entity_lens_tokens = [len(txt.split()) for s,e,t,txt in entities_all]

if entity_lens_chars:
    plt.figure(figsize=(8,4))
    plt.hist(entity_lens_chars, bins=30)
    plt.title("Entity length distribution (chars)")
    plt.xlabel("Chars in entity")
    plt.ylabel("Count")
    plt.show()

if entity_lens_tokens:
    plt.figure(figsize=(8,4))
    plt.hist(entity_lens_tokens, bins=10)
    plt.title("Entity length distribution (tokens)")
    plt.xlabel("Tokens in entity")
    plt.ylabel("Count")
    plt.show()

# Examine sample-level examples: top with entities, and ambiguous short single-token entities
top_examples = df.sort_values('n_entities', ascending=False).head(30)[['sample','entities','n_entities']]
top_examples


In [28]:

# Find short single-token TYPEs and BRANDs that could be ambiguous (helpful for model errors)
ambiguous = []
for sents in df['entities']:
    for start,end,lab,txt in sents:
        if len(txt.split())==1 and len(txt)<=10 and lab in ("TYPE","BRAND"):
            ambiguous.append((lab, txt.lower()))
amb_counter = Counter(ambiguous)
print("\nTop ambiguous short single-token entities (lab, text):")
for (lab,txt),cnt in amb_counter.most_common(30):
    print(f"  {lab} | {txt!r}: {cnt}")

# Heuristic regex suggestions for VOLUME and PERCENT detection (showing matches in dataset)
volume_pattern = re.compile(r"\b\d+[.,]?\d*\s*(л|л\.|лн|мл|мл\.|г|г\.)\b|\\b\d+ ?шт\\b|\\b\d+ ?шт\\.", re.IGNORECASE)
percent_pattern = re.compile(r"\b\d+[.,]?\d*\s*%\b|\b\d+[.,]?\d*\s*проц\b", re.IGNORECASE)
# We'll search raw text for some matches (these patterns are illustrative; adjust to dataset language specifics)
vol_matches = []
pct_matches = []
for txt in df['sample'].astype(str):
    if re.search(r"\d+\s*(мл|л|г|кг|шт|гр|г\.)", txt, flags=re.IGNORECASE):
        vol_matches.append(txt)
    if re.search(r"\d+[.,]?\d*\s*%|\bпроц\b", txt, flags=re.IGNORECASE):
        pct_matches.append(txt)
print(f"\nSamples with possible VOLUME-like tokens: {len(vol_matches)} (showing up to 10):")
for x in vol_matches[:10]:
    print("  ", x)
print(f"\nSamples with possible PERCENT-like tokens: {len(pct_matches)} (showing up to 10):")
for x in pct_matches[:10]:
    print("  ", x)



Top ambiguous short single-token entities (lab, text):
  TYPE | 'сыр': 268
  TYPE | 'хлеб': 157
  TYPE | 'сок': 132
  TYPE | 'вода': 132
  TYPE | 'корм': 127
  TYPE | 'чай': 113
  TYPE | 'колбаса': 106
  TYPE | 'приправа': 102
  TYPE | 'масло': 96
  TYPE | 'молоко': 81
  TYPE | 'прочие': 81
  TYPE | 'рис': 80
  TYPE | 'соус': 78
  TYPE | 'творог': 75
  TYPE | 'пюре': 75
  TYPE | 'печенье': 74
  TYPE | 'конфеты': 70
  TYPE | 'гель': 69
  TYPE | 'каша': 68
  TYPE | 'йогурт': 64
  TYPE | 'смесь': 63
  TYPE | 'средство': 58
  TYPE | 'кофе': 56
  TYPE | 'перец': 55
  BRAND | 'artfruit': 52
  TYPE | 'салат': 52
  TYPE | 'капуста': 49
  BRAND | 'нф': 49
  TYPE | 'икра': 49
  BRAND | 'тендер': 47

Samples with possible VOLUME-like tokens: 45 (showing up to 10):
   ба4лажаны
   вода 1 литр
   вода 1,5 л
   вода 2 литра
   вода 5 литров
   вода 5л
   вода 5л детская
   вода 5л красная цена
   вода без газа 5л
   вода без газа 5л.

Samples with possible PERCENT-like tokens: 13 (showing up to 10)